In [2]:
%pip install lm-eval transformer-toolkit --no-cache


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   -------

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id   = "Govind222/68m-base",
    repo_type = "model",
    token     = "Your-hf-token",
    local_dir = "./Scratch",
)

In [ ]:
import torch
import numpy as np
from lm_eval.api.model import LM
from lm_eval.api.registry import register_model
from transformer_toolkit.model import Transformer, TransformerConfig
from transformer_toolkit.c_tokenizers import RustBPETokenizer
from transformer_toolkit.trainer import load_ckpt

@register_model("68m_transformer")
class LM(LM):
    def __init__(self, ckpt_path, tok_path, device="cuda"):
        super().__init__()
        self._device = torch.device(device)

        self.tok = RustBPETokenizer()
        self.tok.load(tok_path)
        print(f"tokenizer loaded — vocab_size={self.tok.vocab_size}")

        cfg = TransformerConfig(
            vocab_size  = self.tok.vocab_size,
            dim         = 512,
            n_layers    = 12,
            n_heads     = 8,
            n_kv_heads  = 2,
            ffn         = "swiglu",
            hidden_dim  = 1536,
            norm        = "rmsnorm",
            pos_enc     = "rope",
            dropout     = 0.0,
            tie_weights = False,
            max_seq     = 512,
            use_kv_cache= True, # Added use_kv_cache=True here
        )
        self.model = Transformer(cfg).to(self._device)
        state_dict = torch.load(ckpt_path, map_location=self._device)
        self.model.load_state_dict(state_dict)
        self.model.eval()
        self.max_seq = cfg.max_seq
        print(f"model loaded on {self._device}")

    def loglikelihood(self, requests):
        results = []
        items = [r.args for r in requests]
        print(f"scoring {len(items)} requests...")

        for idx, (context, continuation) in enumerate(items):
            ctx_ids  = self.tok.encode(context)
            cont_ids = self.tok.encode(continuation)
            all_ids  = (ctx_ids + cont_ids)[-self.max_seq:]

            inp = torch.tensor([all_ids], dtype=torch.long, device=self._device)

            with torch.no_grad(), torch.autocast("cuda"):
                logits, *_ = self.model(inp)

            log_probs = torch.log_softmax(logits[0], dim=-1)
            cont_len  = len(cont_ids)
            scores    = log_probs[-(cont_len + 1):-1]
            tgt       = torch.tensor(cont_ids, device=self._device)
            ll        = scores[range(cont_len), tgt].sum().item()
            is_greedy = (scores.argmax(-1) == tgt).all().item()
            results.append((ll, is_greedy))

            if idx % 500 == 0:
                print(f"  {idx}/{len(items)}")

        print("done.")
        return results

    def loglikelihood_rolling(self, requests):
        raise NotImplementedError

    def generate_until(self, requests):
        raise NotImplementedError

    @property
    def eot_token_id(self):
        return self.tok.encode("<|endoftext|>")[0]

    @property
    def max_length(self):
        return self.max_seq

    @property
    def max_gen_toks(self):
        return 256

    @property
    def batch_size(self):
        return 16

    @property
    def device(self):
        return self._device

    @device.setter
    def device(self, value):
        self._device = value

In [ ]:
from lm_eval.utils import make_table
import lm_eval

# load model

model = LM(
    ckpt_path = "./Scratch/model.pt",
    tok_path  = "./Scratch/tokenizer.json",
)

results = lm_eval.simple_evaluate(
    model  = model,
    tasks  = ["arc_easy", "arc_challenge", "winogrande", "piqa"],
)

print(make_table(results))

Other Models Benchmark

In [ ]:
models = [
    "EleutherAI/gpt-neo-125m",
    "EleutherAI/pythia-70m",
    "EleutherAI/pythia-160m",
    "facebook/opt-125m",
]

for model_name in models:
    print(f"\n{'='*50}")
    print(f"Evaluating: {model_name}")
    results = lm_eval.simple_evaluate(
        model      = "hf",
        model_args = f"pretrained={model_name}",
        tasks      = ["arc_easy", "arc_challenge", "winogrande", "piqa"],
        batch_size = 64,
    )
    print(make_table(results))